This notebook estimates domestic production domestically consumed for steel-related goods, based on the production data of World Steel Association.

There is a limitation to the methodology applied in this notebook. World Steel Association provides total production of semi-finished and finished steel products in crude steel equivalent. BACI provides export data in tonnes of product directly. Ideally a ratio should thus be applied to account for losses triggered by the transformation of crude steel to products of steel. Since this ratio appears to be quite low (1.1 according to World Steel Association) we choose to disregard and accept this limitation as part of the uncertainty of this estimation.

In [5]:
import pandas as pd
import sqlite3
import json
import country_converter as coco

In [35]:
# load the data from World Steel Association
worldsteeldata = pd.read_excel('exp-2026-05-06_17_09_22.xlsx',index_col=0)
worldsteeldata.index = [coco.convert(i, to='ISO3') for i in worldsteeldata.index]
# world steel data in thousands of tonnes, so convert to tonnes
worldsteeldata *= 1000
worldsteeldata = worldsteeldata.drop(2025, axis=1)

To distribute the broad production data of World steel to the many HS codes, we rely on exports. Thus we extract export data from the full BACI SQL database, but only for relevant HS codes (those pertaining to steel items)

In [16]:
full_baci_conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/baci_trade_data_full_version.db')

query = """
SELECT * FROM [International trade data] 
WHERE 
    -- Iron and Steel (Ingots, Flats, Longs, Wire)
    (cmdCode LIKE '7206%' OR cmdCode LIKE '7207%' OR cmdCode LIKE '7208%' OR 
     cmdCode LIKE '7209%' OR cmdCode LIKE '7210%' OR cmdCode LIKE '7211%' OR 
     cmdCode LIKE '7212%' OR cmdCode LIKE '7213%' OR cmdCode LIKE '7214%' OR 
     cmdCode LIKE '7215%' OR cmdCode LIKE '7216%' OR cmdCode LIKE '7217%' OR 
     cmdCode LIKE '7218%' OR cmdCode LIKE '7219%' OR cmdCode LIKE '7220%' OR 
     cmdCode LIKE '7221%' OR cmdCode LIKE '7222%' OR cmdCode LIKE '7223%' OR 
     cmdCode LIKE '7224%' OR cmdCode LIKE '7225%' OR cmdCode LIKE '7226%' OR 
     cmdCode LIKE '7227%' OR cmdCode LIKE '7228%' OR cmdCode LIKE '7229%')
    OR
    -- Articles of Steel (Tubes, Pipes, Rails, Piling)
    (cmdCode LIKE '7301%' OR cmdCode LIKE '7302%' OR cmdCode LIKE '7303%' OR 
     cmdCode LIKE '7304%' OR cmdCode LIKE '7305%' OR cmdCode LIKE '7306%')
"""

# Execute via pandas
steel_data = pd.read_sql(query, full_baci_conn)

In [20]:
# get imports
import_steel = steel_data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
# get exports
export_steel = steel_data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()
# get net exports
net_exports_steel = (
    export_steel.drop('cmdCode', axis=1).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter']) -
    import_steel.drop('cmdCode', axis=1).rename(columns={'importer':'exporter'}).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter'])
)
# only keep positive net export balances
net_exports_steel = net_exports_steel[net_exports_steel > 0].dropna()

World Steel production data is total production data. We thus exclude exports value from this total production values. And we only keep positive residual production value.

In [36]:
for year in [2021, 2022, 2023, 2024]:
    for country in worldsteeldata.index:
        if country in net_exports_steel.loc[year].index:
            worldsteeldata.loc[country, year] -= net_exports_steel.loc(axis=0)[year, country].iloc[0]

worldsteeldata = worldsteeldata[worldsteeldata > 0]

We need to redistribute the total production of steel product from the World Steel Association to the many different products of steel that are in BACI. We use the country-specific net export ratio of each of these BACI products to estimate their production levels.

In [54]:
net_exports_steel_detailed = (export_steel.set_index(['cmdCode','refYear','exporter']) - 
                              import_steel.rename(columns={'importer':'exporter'}).set_index(['cmdCode','refYear','exporter'])).reset_index()
net_exports_steel_detailed = net_exports_steel_detailed[net_exports_steel_detailed.loc[:,'quantity (t)'] > 0].dropna()

In [71]:
steel_production_data = pd.DataFrame()

for country in worldsteeldata.index:
    for year in worldsteeldata.columns:
        df = net_exports_steel_detailed.loc[(net_exports_steel_detailed.refYear == year) & (net_exports_steel_detailed.exporter == country)].copy()
        df.loc[:, 'quantity (t)'] /= df.loc[:,'quantity (t)'].sum()
        df.loc[:, 'quantity (t)'] *= worldsteeldata.loc[country, year]
        steel_production_data = pd.concat([steel_production_data, df])

steel_production_data = steel_production_data.dropna()

In [ ]:
# some final formatting
steel_production_data.loc[:, 'source'] = 'Data adapted from World Steel Association. Original data taken from https://worldsteel.org/data/annual-production-steel-data.'
steel_production_data.loc[:, 'country_coverage'] = 'complete'
steel_production_data = steel_production_data.rename(columns={'exporter':'producer'})

Add the data to the SQL database inside the "Real production" data table